# Assignment 10B: Credit Card Fraud Detection Using ANN

This notebook implements a deep learning workflow for detecting fraudulent credit card transactions using an Artificial Neural Network (ANN).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow version:', tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
path = kagglehub.dataset_download("quant101/cc-fraud-dataset")
print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "creditcard.csv")

In [ ]:
data = pd.read_csv(csv_path)
print('Dataset shape:', data.shape)
print(data.head().to_string(index=False))
print(f"\nMissing values: {data.isnull().sum().sum()}")
print(f"Duplicate rows: {data.duplicated().sum()}")
print(f"""\nClass distribution:\n{data['Class'].value_counts().to_string()}""")
print(f"\nClass ratio:\n{data['Class'].value_counts(normalize=True).to_string()}")

plt.figure(figsize=(8, 5))
sns.countplot(x='Class', data=data, palette='Set2')
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

print(data.describe().T.head())

In [ ]:
data = data.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates:', data.shape)

features = data.drop(columns=['Class'])
labels = data['Class']

X_train, X_test, y_train, y_test = train_test_split(
    features, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Train class ratio: {y_train.value_counts(normalize=True).to_dict()}')
print(f'Test class ratio: {y_test.value_counts(normalize=True).to_dict()}')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

print('Scaling complete.')

## Handling Class Imbalance

The dataset is highly imbalanced with fraud cases being only about 0.17% of all transactions. Class weights are used during training to give more attention to the minority class.

In [ ]:
from collections import Counter

class_counts = Counter(y_train)
print('Training class counts:', dict(class_counts))

class_weight = {0: 1.0, 1: (class_counts[0] / max(class_counts[1], 1))}
print('Class weights:', class_weight)

## ANN Model Experiments

In [ ]:
def create_ann_model(h1=64, h2=32, act='relu', opt='adam', drop=0.2):
    net = keras.Sequential([
        layers.Input(shape=(X_train_scaled.shape[1],)),
        layers.Dense(h1, activation=act),
        layers.Dropout(drop),
        layers.Dense(h2, activation=act),
        layers.Dropout(drop / 2),
        layers.Dense(1, activation='sigmoid')
    ])
    net.compile(
        optimizer=opt,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return net

experiments = [
    {'tag': 'Exp_1', 'act': 'relu', 'opt': 'adam', 'epochs': 25, 'h1': 64, 'h2': 32},
    {'tag': 'Exp_2', 'act': 'tanh', 'opt': 'sgd', 'epochs': 25, 'h1': 48, 'h2': 24},
    {'tag': 'Exp_3', 'act': 'relu', 'opt': 'rmsprop', 'epochs': 30, 'h1': 80, 'h2': 40},
    {'tag': 'Exp_4', 'act': 'sigmoid', 'opt': 'adam', 'epochs': 25, 'h1': 64, 'h2': 32},
]

experiment_results = []

for exp in experiments:
    net = create_ann_model(
        h1=exp['h1'],
        h2=exp['h2'],
        act=exp['act'],
        opt=exp['opt']
    )
    print(f"\nTraining {exp['tag']}... act={exp['act']}, opt={exp['opt']}, epochs={exp['epochs']}")
    hist = net.fit(
        X_train_scaled,
        y_train,
        validation_split=0.1,
        epochs=exp['epochs'],
        batch_size=512,
        class_weight=class_weight,
        verbose=0
    )

    probs = net.predict(X_test_scaled, verbose=0).ravel()
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    cm = confusion_matrix(y_test, preds)

    experiment_results.append({
        'Experiment': exp['tag'],
        'Activation': exp['act'],
        'Optimizer': exp['opt'],
        'Epochs': exp['epochs'],
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'Confusion Matrix': cm,
        'History': hist.history
    })

    print(classification_report(y_test, preds, digits=4, zero_division=0))
    print('Confusion Matrix:', cm)

## Experiment Comparison

In [ ]:
results_df = pd.DataFrame([
    {
        'Experiment': r['Experiment'],
        'Activation': r['Activation'],
        'Optimizer': r['Optimizer'],
        'Epochs': r['Epochs'],
        'Accuracy': r['Accuracy'],
        'Precision': r['Precision'],
        'Recall': r['Recall'],
        'F1': r['F1']
    }
    for r in experiment_results
])

print(results_df.to_string(index=False))

best_row = results_df.sort_values(['Recall', 'F1', 'Accuracy'], ascending=False).iloc[0]
print(f"\nBest experiment (by Recall/F1): {best_row['Experiment']}")

## Training Loss Visualization

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
exp_labels = [r['Experiment'] for r in experiment_results]

for ax, res in zip(axes, experiment_results):
    ax.plot(res['History']['loss'], label='Training Loss')
    ax.plot(res['History']['val_loss'], label='Validation Loss')
    ax.set_title(f"{res['Experiment']}: {res['Activation']}-{res['Optimizer']}")
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()

plt.tight_layout()
plt.show()

## Final Predictions with Best Model

In [ ]:
best = max(experiment_results, key=lambda x: (x['Recall'], x['F1'], x['Accuracy']))
best_cfg = next(e for e in experiments if e['tag'] == best['Experiment'])

final_net = create_ann_model(
    h1=best_cfg['h1'],
    h2=best_cfg['h2'],
    act=best_cfg['act'],
    opt=best_cfg['opt']
)

final_net.fit(
    X_train_scaled,
    y_train,
    validation_split=0.1,
    epochs=best_cfg['epochs'],
    batch_size=512,
    class_weight=class_weight,
    verbose=0
)

final_probs = final_net.predict(X_test_scaled, verbose=0).ravel()
final_preds = (final_probs >= 0.5).astype(int)

print('Final Evaluation:')
print(classification_report(y_test, final_preds, digits=4, zero_division=0))

print('Sample Predictions (Actual vs Predicted):')
for i in range(10):
    print(f'  Index {i}: Actual={y_test[i]}, Predicted={final_preds[i]}, Prob={final_probs[i]:.4f}')

print("\n0 = Legitimate, 1 = Fraudulent")

## Conclusion

Multiple ANN configurations with different activation functions and optimizers were tested for credit card fraud detection. The best model was selected based on Recall and F1-score rather than just Accuracy, because in fraud detection, catching fraudulent transactions is more important than overall accuracy.